# Analyze Param Sweep

Set of visualization tools to check param sweep runs.

In [ ]:
%matplotlib widget

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
RUN_DIR = "/home/user/.cache/nanorepro/runs"
SWEEP_PREFIX = "sweep_"    # only consider run directories starting with 'sweep_'
assert os.path.exists(RUN_DIR)

In [ ]:
log_dirs = [d for d in os.listdir(RUN_DIR) if d.startswith(SWEEP_PREFIX) and os.path.isdir(os.path.join(RUN_DIR, d))]
log_files = [os.path.join(RUN_DIR, log_dir, "train_log_rank0.jsonl") for log_dir in log_dirs]
log_files = sorted(log_files)
print(log_files)

In [ ]:
# Load logs
log_bpb_eval = []
for log_file in log_files:
    with open(log_file, "r") as f:
        lines = f.readlines()
        user_config = json.loads(lines[0])
        assert user_config["event"] == "user_config"
        model_config = json.loads(lines[1])
        assert model_config["event"] == "model_config"
        params_counts = json.loads(lines[2])
        assert params_counts["event"] == "params_counts"
        training_hyperparameters = json.loads(lines[3])
        assert training_hyperparameters["event"] == "training_hyperparameters"
        
        
        depth = user_config["depth"]
        matrix_lr = user_config["matrix_lr"]
        embedding_lr = user_config["embedding_lr"]
        warmdown_ratio = user_config["warmdown_ratio"]
        total_batch_size = user_config["total_batch_size"]
        weight_decay = user_config["weight_decay"]
        run = user_config["run"]
        run_group = '_'.join(run.split("_")[:-1])

        single_run_log_objects = [json.loads(line) for line in lines]
        single_run_bpb_eval_objects = [obj for obj in single_run_log_objects if obj['event'] == 'bpb_eval']
        final_bpb_eval_object = single_run_bpb_eval_objects[-1]
        final_bpb_eval = final_bpb_eval_object['val/bpb']

        log_bpb_eval.append({
            'val/bpb': final_bpb_eval,
            'depth': depth,
            'matrix_lr': matrix_lr,
            'embedding_lr': embedding_lr,
            'warmdown_ratio': warmdown_ratio,
            'batch_size': total_batch_size,
            'weight_decay': weight_decay,
            'run': run,
            'run_group': run_group,
        })
        print(f"{log_file}: {len(lines)} lines")

In [ ]:
df = pd.DataFrame(log_bpb_eval)

In [ ]:
# --------------------------------------------
#    Matrix LR vs BPB (for a fixed depth)
df_mlr = df[df['run_group'].str.contains("mlr")]
plt.figure(figsize=(5, 5))
plt.scatter(df_mlr['matrix_lr'], df_mlr['val/bpb'], label="mlr")
plt.xticks(sorted(df_mlr['matrix_lr'].unique()))
plt.title(f"Matrix LR vs BPB (depth={df_mlr['depth'].iloc[0]})")
plt.xlabel("Matrix LR")
plt.ylabel("BPB")
plt.grid(True, which="both", lw=0.5, ls="--")
plt.savefig("analyze_param_sweep_matrix_lr_vs_bpb.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# --------------------------------------------
#    Embedding LR vs BPB (for a fixed depth)
df_elr = df[df['run_group'].str.contains("embd_lr")]

plt.figure(figsize=(5, 5))
plt.scatter(df_elr['embedding_lr'], df_elr['val/bpb'], label="elr")
plt.xticks(sorted(df_elr['embedding_lr'].unique()))
plt.title(f"Embedding LR vs BPB (depth={df_elr['depth'].iloc[0]})")
plt.xlabel("Embedding LR")
plt.ylabel("BPB")
plt.grid(True, which="both", lw=0.5, ls="--")
plt.savefig("analyze_param_sweep_embedding_lr_vs_bpb.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# --------------------------------------------
#    Warmdown Ratio vs BPB (for a fixed depth)
df_wr = df[df['run_group'].str.contains("warmdown")]

plt.figure(figsize=(5, 5))
plt.scatter(df_wr['warmdown_ratio'], df_wr['val/bpb'], label="wr")
plt.xticks(sorted(df_wr['warmdown_ratio'].unique()))
plt.title(f"Warmdown Ratio vs BPB (depth={df_wr['depth'].iloc[0]})")
plt.xlabel("Warmdown Ratio")
plt.ylabel("BPB")
plt.grid(True, which="both", lw=0.5, ls="--")
plt.savefig("analyze_param_sweep_warmdown_ratio_vs_bpb.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# --------------------------------------------
#    Batch Size vs BPB (for a fixed depth)
df_bs = df[df['run_group'].str.contains("batch_size")]

plt.figure(figsize=(5, 5))
plt.scatter(df_bs['batch_size'], df_bs['val/bpb'], label="bs")
plt.xticks(sorted(df_bs['batch_size'].unique()))
plt.title(f"Batch Size vs BPB (depth={df_bs['depth'].iloc[0]})")
plt.xlabel("Batch Size")
plt.ylabel("BPB")
plt.grid(True, which="both", lw=0.5, ls="--")
plt.savefig("analyze_param_sweep_batch_size_vs_bpb.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# --------------------------------------------
#    Weight Decay vs BPB (for a fixed depth)
df_wd = df[df['run_group'].str.contains("weight_decay")]

plt.figure(figsize=(5, 5))
plt.scatter(df_wd['weight_decay'], df_wd['val/bpb'], label="wd")
plt.xticks(sorted(df_wd['weight_decay'].unique()))
plt.title(f"Weight Decay vs BPB (depth={df_wd['depth'].iloc[0]})")
plt.xlabel("Weight Decay")
plt.ylabel("BPB")
plt.grid(True, which="both", lw=0.5, ls="--")
plt.savefig("analyze_param_sweep_weight_decay_vs_bpb.png", dpi=200, bbox_inches="tight")
plt.show()